<a href="https://colab.research.google.com/github/Saleha65/Machine-Learning-by-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saleha65/Machine-Learning-by-Flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
from getpass import getpass

os.environ["HF_TOKEN"] = getpass("Paste your HF read token: ")

Paste your HF read token: ··········


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [4]:
from datasets import load_dataset
import pandas as pd

ds_fact = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    data_files="**/month=2026-03/*.parquet",
    download_mode="force_redownload"
)
fact_df = ds_fact["train"].to_pandas()

ds_dim = load_dataset("FlyRank/internship-warehouse", "dim_content")
dim_df = ds_dim["train"].to_pandas()

fact_df["report_date"] = pd.to_datetime(fact_df["report_date"])
month_df = fact_df.copy()

print(month_df.shape)
month_df.head()

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

(9841378, 30)


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Unit of analysis: one row = one content page (content_id) for one client, on one report_date.
Time window: month = 2026-03 (mid-panel month, filtered from report_date).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [5]:
fields = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_sum_position"],
    "label": ["refresh_priority_score"],  # proxy — abhi dataset mein nahi, hum khud banayenge baad mein
    "context": ["content_hash_id", "client_hash_id", "report_date"],
    "excluded": ["rows_where_gsc_data_available_is_false"]
}
fields

{'feature': ['gsc_impressions', 'gsc_clicks', 'gsc_sum_position'],
 'label': ['refresh_priority_score'],
 'context': ['content_hash_id', 'client_hash_id', 'report_date'],
 'excluded': ['rows_where_gsc_data_available_is_false']}

Feature: days_since_last_update, impressions, clicks, avg_position, sessions
Label/proxy: refresh_priority_score — built from staleness + declining avg_position + declining sessions
Context: content_hash_id, client_hash_id, report_date
Excluded: pages with zero impressions across the whole month — no signal to rank on

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# 1. Grain check
dupes = month_df.groupby(["content_hash_id", "client_hash_id", "report_date"]).size()
print("Grain violations:", (dupes > 1).sum())

# 2. Row count + date span
print("Total rows:", len(month_df))
print("Date span:", month_df["report_date"].min(), "to", month_df["report_date"].max())

# 3. Availability — real IS TRUE filter using the actual column
available = month_df[month_df["gsc_data_available"] == True]
print(f"{len(available)} of {len(month_df)} rows have gsc_data_available == True")

Grain violations: 0
Total rows: 9841378
Date span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
3611061 of 9841378 rows have gsc_data_available == True


Grain check: 0 violations found — confirms one row = one page per client per day. Row count: 9,841,378 rows spanning 2026-03-01 to 2026-03-31. Availability: 3,611,061 of 9,841,378 rows have gsc_data_available == True — the rest have no real GSC data recorded for that day, not zero activity.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [7]:
pass

This slice cannot tell you: whether a refresh actually caused better rankings (correlational, not causal); performance for clients or pages outside this warehouse; or true behavior in months with GSC-only early data or overlapping windows.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.